# Lab 2 - Notebook 2: Baseline Vector Search

**Goal:** search the corpus using vector similarity alone, and see where it falls short.

**Concept:** we embed the query the same way we embedded the documents, then find
the documents whose vectors are closest (cosine similarity). This is *semantic search*:
it already beats keyword matching. But as you will see, the best answer is not always
ranked first, and that is the gap Rerank closes in Notebook 3.


## 1. Load corpus, vectors, helpers

Each notebook runs in its own fresh session, so we re-import the helpers and re-load
the corpus and the vectors we saved in Notebook 1. Nothing carries over automatically
between notebooks; the saved `doc_vectors.npy` file is how the data is passed along.


In [ ]:
from cohere_helpers import embed_texts, cosine_search
import json, numpy as np

with open('support_corpus.json', 'r', encoding='utf-8') as f:
    corpus = json.load(f)
doc_vectors = np.load('doc_vectors.npy')
doc_texts = [f"{d['title']}. {d['content']}" for d in corpus]
print(f'{len(corpus)} docs, vectors {doc_vectors.shape}')


## 2. A helper to show results nicely

This small function just prints the top results in a readable way. We define it here
(rather than in the shared module) because it is specific to how this notebook displays
search results.


In [ ]:
def show(results, k=5):
    for rank, (i, score) in enumerate(results[:k], 1):
        print(f"{rank}. [{score:.3f}] {corpus[i]['title']}")


## 3. Search with a single query

We embed the query with `input_type='search_query'`, then rank all documents
by cosine similarity. Look at the top 5: are they actually the best answers?


In [ ]:
query = 'I cannot log in after changing my password'

q_vec = embed_texts([query], input_type='search_query')[0]
vec_results = cosine_search(q_vec, doc_vectors, top_k=10)

print(f'Query: {query}')
print('Top 5 by vector search:')
show(vec_results)


## 4. Try the demo queries

These five queries are designed to expose where pure vector search struggles.
For several of them, a document that *shares words* with the query ranks high,
while the *actually correct* answer (worded differently) ranks lower.

If the output is long, you may see a message that it is truncated; click
"scrollable element" to view all of it. That is normal.


In [ ]:
demo_queries = [
    'I cannot log in after changing my password',
    'Why is my bill higher than expected this month',
    'My deployment keeps failing',
    'How do I get my data out of the platform',
    'I keep getting rate limited with 429 errors',
]

for q in demo_queries:
    qv = embed_texts([q], input_type='search_query')[0]
    res = cosine_search(qv, doc_vectors, top_k=5)
    print(f'\nQuery: {q}')
    show(res, k=3)


## Your turn: add your own query

Open any file in the corpus (or scroll the documents you loaded) and pick a topic.
Then add a question about that topic to the end of the `demo_queries` list above,
phrased in your own words, and re-run the cell. See where the matching document lands.

Keep your query in mind; you will run it through Rerank in the next notebook and
compare.


## Checkpoint

Vector search is good, but not perfect. For some queries the top result is a
near-miss that shares vocabulary rather than the true answer.

In **Notebook 3: Rerank**, we keep these same vector-search results and let
Rerank-v4 reorder them. Watch the correct answers climb, and the scores sharpen.
